# Running Ollama on Google Colab

This notebook installs and runs [Ollama](https://ollama.com) inside a Google Colab environment.

⚠️ **Requirements & Notes**
- Works best on **Colab Pro/Pro+** with a GPU runtime.
- Each Colab session is ephemeral – you’ll need to rerun setup when the runtime resets.
- Large models may not fit into Colab’s limited disk / VRAM.

---
## 1. Check GPU
Run the cell below to see if a GPU is available.


In [1]:
# Check if a GPU is available
!nvidia-smi || echo 'No GPU available. You may be on a CPU-only runtime.'

Mon Dec  8 07:24:33 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Ollama
This downloads and installs the Ollama CLI for Linux (the OS used by Colab).


In [2]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh


>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## 3. Start the Ollama server
Ollama runs as a background server process. We start it in the background.

The `&` at the end runs the server in the background so the notebook can keep going.


In [3]:
# Start Ollama server in the background
import subprocess, time

server = subprocess.Popen([
    'ollama', 'serve'
])
print('Ollama server started (background process).')
time.sleep(5)
print('Waited 5 seconds for the server to initialize.')

Ollama server started (background process).
Waited 5 seconds for the server to initialize.


## 4. Pull a model
Here we pull **Llama 3.1** as an example. You can replace `llama3.1` with any other Ollama model name (e.g. `mistral`, `qwen2.5`, `phi3`, `gemma2`).


In [4]:
# Download a model (this may take a while, depending on size and connection)
!ollama pull deepseek-r1:1.5b

## 5. Simple one-off prompt from the shell
You can run a one-off prompt directly from the shell using `ollama run`.


In [5]:
!ollama run llama3.1 "Explain retrieval-augmented generation in simple terms."


Retrieval-augmented generation (RAG) is a type of artificial intelligence model that combines two powerful techniques to generate text:

1. **Retrieval**: This part uses a large database or corpus of existing texts to find relevant information related to the input prompt.
2. **Generation**: This part uses AI algorithms to create new, coherent text based on the retrieved information.

Here's how it works in simple terms:

**Step 1: Retrieval**

* The model receives an input prompt from a user (e.g., "Write a story about a character who learns to fly").
* It searches through a massive database of texts and finds relevant pieces that match the prompt, like:
	+ A Wikipedia article on human flight
	+ A short story about a superhero with flying abilities
	+ Some scientific papers about aerodynamics

**Step 2: Generation**

* The model takes these retrieved pieces as input and uses AI algorithms to combine them into new text.
* It might generate text that is similar in style or structure to 

## 6. Interactive use from Python
Below is a simple Python helper to send prompts to Ollama and get back responses.

We’ll use the `ollama` CLI via `subprocess`. For more advanced usage, you could expose the HTTP API, but this keeps it simple.


In [6]:
import subprocess

def ollama_chat(model: str, prompt: str) -> str:
    """Send a single prompt to an Ollama model and return the response as text."""
    result = subprocess.run(
        ['ollama', 'run', model],
        input=prompt,
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Ollama error: {result.stderr}")
    return result.stdout

response = ollama_chat('llama3.1', 'Give me a short bullet list of three use cases for Ollama in Colab.')
print(response)

I couldn't find any information on "Ollama" or its relation to Google Colab. If you are referring to Llama, here is a short bullet list of three potential use cases:

*   **Automated Code Review and Debugging:** Llama can help identify bugs in code and suggest improvements. It can be used to review the code for syntax errors, logical consistency, and adherence to best practices.
*   **Code Generation and Augmentation:** Llama can assist with generating new code or augmenting existing code by providing suggestions based on its understanding of programming concepts and syntax. This can help speed up the development process and improve productivity.
*   **Exploratory Data Analysis and Visualization:** Llama can help analyze data, generate visualizations, and provide insights into complex datasets. It can be used to identify trends, relationships, and patterns in the data, making it easier to draw meaningful conclusions.

Please note that these are just potential use cases and actual capab

## 7. (Optional) Very simple RAG-style example
This is a **toy** retrieval-augmented generation example using Python only. It:
1. Stores a few text chunks in memory.
2. Retrieves the most relevant chunk with a crude cosine similarity over sentence embeddings (from `sentence-transformers`).
3. Sends both the retrieved context and the question to the Ollama model.

You can expand this later for your research prototype.


In [ ]:
!pip install -q sentence-transformers

In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np
import textwrap
import subprocess

# 1. Example corpus (replace with your own documents)
documents = [
    "Retrieval-augmented generation (RAG) combines a retriever and a generator.",
    "Ollama can run local language models and expose them via a simple CLI.",
    "Vector databases like FAISS or Chroma are often used for efficient similarity search.",
]

# 2. Embed documents
embedder = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedder.encode(documents, convert_to_numpy=True)

def retrieve_context(query: str, k: int = 1):
    q_emb = embedder.encode([query], convert_to_numpy=True)[0]
    # Compute cosine similarity
    sims = doc_embeddings @ q_emb / (
        np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(q_emb) + 1e-10
    )
    top_idx = np.argsort(-sims)[:k]
    return [documents[i] for i in top_idx]

def ollama_rag(model: str, question: str) -> str:
    context_chunks = retrieve_context(question, k=2)
    context = "\n".join(f"- {c}" for c in context_chunks)
    prompt = textwrap.dedent(f"""
    You are an assistant with access to the following context:
    {context}

    Using ONLY this context when possible, answer the question:
    {question}
    """)
    result = subprocess.run(
        ['ollama', 'run', model],
        input=prompt,
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Ollama error: {result.stderr}")
    return result.stdout

answer = ollama_rag('llama3.1', 'How does RAG use retrieval and generation together?')
print(answer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In Retrieval-Augmented Generation (RAG), the retriever uses a vector database to efficiently search for relevant documents based on user input. The generator then takes these retrieved documents as input and generates a response.

Here's a step-by-step overview of how RAG combines retrieval and generation:

1. **Retriever**: Given a user's query or prompt, the retriever searches a large corpus of text (e.g., Wikipedia articles) using a vector database like FAISS or Chroma. The database indexes the semantic meaning of each document as a dense vector representation.
2. **Similarity Search**: The retriever performs a similarity search between the input query and the indexed vectors in the database. This yields a set of top-ranked documents that are most relevant to the user's query.
3. **Generator**: The generator takes the retrieved documents as input, along with the original user query or prompt. It uses these inputs to generate a response that is coherent and accurate.

By separating t

## 8. Cleanup (optional)
If you want to stop the Ollama server manually, you can terminate the process. In Colab, the easiest way is often just to **restart the runtime**, but you can also run the cell below.


In [8]:
try:
    server.terminate()
    print('Terminated Ollama server process.')
except NameError:
    print('Server variable not found; maybe the runtime was restarted or the server was never started in this session.')

Terminated Ollama server process.
